# Fase 1 - Computacion paralela con Dask

**Bootcamp:** Python PY4  
**Modulo 4:** Big Data y MLOps  
**Practica: NoSQL - Integracion de Python y Procesamiento Paralelo**

## De que se trata esta sesion

Cuando trabaja con Big Data, es comun encontrarnos con
archivos que pesan varios gigabytes (logs de servidores, transacciones bancarias de
varios anios, imagenes satelitales, etc.). Cuando el dataset supera la RAM disponible,
herramientas como Pandas dejan de ser viables: simplemente no hay espacio en memoria
para cargar todo de una vez.

En esta sesion vamos a:

1. Generar un dataset sintetico de transacciones bancarias, particionado en varios
   archivos CSV (para simular un escenario de "Big Data").
2. Cargar y procesar ese dataset con **Pandas**, de forma secuencial, y medir cuanto
   tiempo toma.
3. Cargar y procesar el mismo dataset con **Dask**, usando *lazy evaluation* y
   procesamiento en paralelo, y comparar los tiempos.
4. Practicar filtros y agregaciones avanzadas con `dask.dataframe`.
5. Revisar los errores mas comunes al trabajar con Dask.
6. Exportar un reporte final que vamos a reutilizar en la Sesion 2 (MongoDB).

Este es el mismo ejercicio conceptual de "Analisis de Transacciones Financieras" que
vimos en el material de la semana, pero llevado a la practica paso a paso.


#### PySpark vs Dask

* **PySpark:** Framework para procesar **Big Data** de forma distribuida. Ideal para grandes datasets y pipelines de ETL (Extract Transform, Load).
* **Dask:** Extension del ecosistema de **Python** para ejecutar Pandas, NumPy y otras operaciones en paralelo. Ideal para escalar workflows de Python.

**Ejemplo:**  
Si trabajas con varios TB de datos y necesitas construir un pipeline ETL. --> PySpark   
Si tienes un workflow de Pandas que ya no cabe en memoria --> Dask



#### Procesamiento paralelo vs distribuido.

- Paralelo: varias tareas se ejecutan al mismo tiempo, normalmente usando varios CPU cores.  
```
CPU
├── Core 1 → Task A
├── Core 2 → Task B
├── Core 3 → Task C
└── Core 4 → Task D
```

- Distribuido: el trabajo se reparte entre varias máquinas o nodes.
```
Cluster
├── Machine 1 → Data A
├── Machine 2 → Data B
├── Machine 3 → Data C
└── Machine 4 → Data D
```

## 1. Preparacion del entorno

Necesitamos `pandas`, `numpy`, y `dask`.

En Google Colab, tenemos que instalar Dask.


In [ ]:
# Instalar Dask (y pyarrow, que Dask usa por dentro)
!pip install dask[dataframe] pyarrow # "[dataframe] " = adicionalmente instala las dependencias necesarias para Dask DataFrame.


In [ ]:
import time # Medir tiempos y trabajar con fechas/horas.
import glob # Buscar archivos usando patrones, por ejemplo *.csv
import os

import numpy as np
import pandas as pd
import dask.dataframe as dd

# Semilla para que el dataset sintetico sea reproducible entre estudiantes
np.random.seed(42)



## 2. Generar el dataset sintetico: transacciones bancarias

Vamos a simular un banco que exporta sus transacciones en varios archivos CSV
(uno por lote de procesamiento), tal como pasaria en un entorno real donde los datos
llegan particionados en vez de en un solo archivo gigante.

Cada fila va a representar una transaccion con las siguientes columnas:

- `id_transaccion`: identificador unico.
- `fecha`: fecha de la transaccion durante 2024.
- `monto`: monto de la transaccion en dolares.
- `region`: departamento de El Salvador donde ocurrio la transaccion.
- `canal`: canal usado (app movil, cajero automatico, sucursal, web, POS).
- `tipo_cliente`: tipo de cliente (personal, empresarial, pyme).

Vamos a generar varios archivos (`transacciones_2024_part1.csv`, `part2.csv`, etc.)
para que Dask los pueda leer como si fueran particiones de un mismo dataset grande.


In [ ]:
DATA_DIR = "data"

os.makedirs(DATA_DIR, exist_ok=True)

departamentos = [
    "San Salvador", "Santa Ana", "San Miguel", "La Libertad", "Sonsonate",
    "Usulutan", "La Paz", "Cuscatlan", "Ahuachapan", "La Union",
]

escalas_region = {
    "San Salvador": 2500,
    "Santa Ana": 1800,
    "San Miguel": 2200,
    "La Libertad": 2800,
    "Sonsonate": 1600,
    "Usulutan": 1400,
    "La Paz": 1300,
    "Cuscatlan": 2100,
    "Ahuachapan": 1100,
    "La Union": 1000,

}

canales = ["app_movil", "cajero_automatico", "sucursal", "web", "pos"]
tipos_cliente = ["personal", "empresarial", "pyme"]

N_ARCHIVOS = 6
FILAS_POR_ARCHIVO = 800_000  # con 6 archivos, el dataset completo ronda los 4.8 millones de filas

#N_ARCHIVOS = 20
#FILAS_POR_ARCHIVO = 1_000_000  # 20 millones de filas

#N_ARCHIVOS = 30
#FILAS_POR_ARCHIVO = 1_000_000  # 30 millones de filas

In [ ]:
def generar_archivo(indice):

    n = FILAS_POR_ARCHIVO  # cantidad de filas por archivo

    inicio_id = indice * n  # calculamos el ID inicial para este archivo

    regiones = np.random.choice(departamentos, n)  # Asignamos una región aleatoria a cada transacción

    escalas = np.array([escalas_region[r] for r in regiones])  # Obtenemos la escala correspondiente a cada región

    df = pd.DataFrame({
        "id_transaccion": np.arange(inicio_id, inicio_id + n),  # Generamos IDs consecutivos
        "fecha": pd.to_datetime("2024-01-01") + pd.to_timedelta(  # Creamos fechas a partir del 1 de enero de 2024
            np.random.randint(0, 365, n), unit="D"  # Generamos un número aleatorio de días para cada transacción
        ),
        "monto": np.round(np.random.exponential(scale=escalas) + 5, 2),  # Generamos montos usando una distribución exponencial según la región
        "region": regiones,  # Asignamos la región generada a cada transacción
        "canal": np.random.choice(canales, n),  # Asignamos un canal aleatorio a cada transacción
        "tipo_cliente": np.random.choice(tipos_cliente, n, p=[0.6, 0.25, 0.15]),  # Asignamos el tipo de cliente según las probabilidades indicadas
    })
    ruta = os.path.join(DATA_DIR, f"transacciones_2024_part{indice + 1}.csv")  # Construimos la ruta y el nombre del archivo
    df.to_csv(ruta, index=False)  # Guardamos el DataFrame como CSV sin incluir el índice
    return ruta  # Retornamos la ruta del archivo generado


t0 = time.time()  # Guardamos el tiempo inicial para medir cuánto tarda el proceso
rutas_generadas = [generar_archivo(i) for i in range(N_ARCHIVOS)]  # Generamos todos los archivos y guardamos sus rutas

print(f"Se generaron {N_ARCHIVOS} archivos en {time.time() - t0:.1f} segundos")  # Mostramos la cantidad de archivos y el tiempo total de generación
for r in rutas_generadas:  # Recorremos las rutas de los archivos generados
    print(" -", r, f"({os.path.getsize(r) / 1e6:.1f} MB)")  # Mostramos la ruta y el tamaño de cada archivo en MB

Se generaron 6 archivos en 36.8 segundos
 - data/transacciones_2024_part1.csv (42.9 MB)
 - data/transacciones_2024_part2.csv (43.6 MB)
 - data/transacciones_2024_part3.csv (43.8 MB)
 - data/transacciones_2024_part4.csv (43.8 MB)
 - data/transacciones_2024_part5.csv (43.8 MB)
 - data/transacciones_2024_part6.csv (43.8 MB)


## 3. Enfoque secuencial: cargar todo con Pandas

Primero vamos a usar la forma "tradicional": leer todos los
archivos con Pandas, concatenarlos en un solo DataFrame que vive completo en RAM, y
despues calcular una agregacion.

El objetivo del analisis es: **para cada region, calcular el monto promedio de
las transacciones de alto valor (monto mayor a 3000 dolares)**. Esto es un indicador
tipico de analisis de riesgo bancario.

Vamos a medir el tiempo total que toma este enfoque.


In [ ]:
t0 = time.time() # Guardamos el tiempo inicial

archivos = sorted(glob.glob(os.path.join(DATA_DIR, "transacciones_2024_part*.csv"))) # Buscamos y ordenamos todos los archivos CSV que coinciden con el patrón
lista_dataframes = [pd.read_csv(archivo) for archivo in archivos] # Leemos cada archivo CSV y lo convertimos en un DataFrame
df_pandas = pd.concat(lista_dataframes, ignore_index=True) # Unimos todos los DataFrames en uno solo y reiniciamos el índice

alto_valor = df_pandas[df_pandas["monto"] > 3000] # Filtramos las transacciones cuyo monto es mayor a 3000
promedio_por_region_pandas = alto_valor.groupby("region")["monto"].mean().sort_values(ascending=False) # Calculamos el monto promedio por región y ordenamos de mayor a menor

tiempo_pandas = time.time() - t0 # Calculamos el tiempo total que tomó ejecutar el procesamiento con Pandas

print(f"Tiempo con Pandas: {tiempo_pandas:.2f} segundos") # Mostramos el tiempo de ejecución en segundos
print(f"Filas totales cargadas en memoria: {len(df_pandas):,}") # Mostramos la cantidad total de filas cargadas en memoria
promedio_por_region_pandas


Tiempo con Pandas: 36.00 segundos
Filas totales cargadas en memoria: 28,800,000


,monto
region,
La Libertad,5799.818216
San Salvador,5504.656368
San Miguel,5196.225108
Cuscatlan,5100.038060
Santa Ana,4802.109659
Sonsonate,4599.266547
Usulutan,4403.447877
La Paz,4302.411255
Ahuachapan,4098.452143


**Que acaba de pasar:** Pandas cargo *todos* los archivos completos en la memoria RAM
antes de poder hacer cualquier calculo. Con 6 archivos de este tamanio todavia es
manejable, pero si el dataset fuera de 50 GB y tu maquina tuviera 16 GB de RAM, este
codigo fallaria con un error de memoria (`MemoryError`).

Ese es exactamente el problema que Dask resuelve.


## 4. Enfoque paralelo: el mismo analisis con Dask

`dask.dataframe` usa **lazy evaluation** (evaluacion perezosa): cuando llamamos a
`dd.read_csv`, Dask *no* lee los datos todavia. Solo construye un "mapa" de como leer
los archivos y en cuantas particiones dividir el trabajo. Las operaciones que
definimos despues (filtros, `groupby`, etc.) tampoco se ejecutan de inmediato: se van
agregando a un **grafo de tareas**.

El calculo real solo ocurre cuando llamamos a `.compute()`. En ese momento, Dask
reparte las particiones entre los nucleos disponibles de tu CPU y las procesa en
paralelo, combinando los resultados parciales al final.

Vamos a repetir exactamente el mismo analisis (transacciones de alto valor por
region) pero con Dask.


In [ ]:
t0 = time.time()

df_dask = dd.read_csv(os.path.join(DATA_DIR, "transacciones_2024_part*.csv")) # Creamos un Dask DataFrame a partir de los archivos CSV sin leerlos todavía
print(f"Numero de particiones que uso Dask: {df_dask.npartitions}")


Numero de particiones que uso Dask: 30


Hasta aqui, nada se ha leido de disco todavia: solo se definio el plan de lectura.

In [ ]:
# Definimos la operacion (sigue sin ejecutarse)
alto_valor_dask = df_dask[df_dask["monto"] > 3000] # Filtramos las transacciones cuyo monto es mayor a 3000
promedio_por_region_dask = alto_valor_dask.groupby("region")["monto"].mean() # Calculamos el monto promedio por región

print(type(promedio_por_region_dask))
# Si imprimes la variable directamente, vas a ver un objeto Dask, no los datos.
# El calculo real se dispara con .compute()


<class 'dask.dataframe.dask_expr._collection.Series'>


In [ ]:
t0 = time.time()

resultado_dask = promedio_por_region_dask.compute().sort_values(ascending=False)  # Ejecutamos el cálculo de Dask y ordenamos los resultados de mayor a menor

tiempo_dask = time.time() - t0  # Calculamos el tiempo total de ejecución, incluyendo la lectura y el procesamiento de los datos

print(f"Tiempo con Dask (incluyendo lectura + compute): {tiempo_dask:.2f} segundos")  # Mostramos el tiempo total que tomó Dask
resultado_dask


Tiempo con Dask (incluyendo lectura + compute): 27.00 segundos


,monto
region,
La Libertad,5799.818216
San Salvador,5504.656368
San Miguel,5196.225108
Cuscatlan,5100.038060
Santa Ana,4802.109659
Sonsonate,4599.266547
Usulutan,4403.447877
La Paz,4302.411255
Ahuachapan,4098.452143


## 5. Comparando los dos enfoques

Vamos a poner los tiempos lado a lado.

**Nota importante:**
La leccion clave **no es "Dask siempre es mas rapido"**, sino: *Dask te permite
procesar datasets que no caben en RAM, dividiendo el trabajo en particiones que se
procesan de forma independiente y en paralelo.* La velocidad es una consecuencia de
eso, no el punto central.


In [ ]:
comparacion = pd.DataFrame({ # Creamos un DataFrame para comparar los tiempos de ejecución
    "enfoque": ["Pandas (secuencial)", "Dask (paralelo)"],
    "tiempo_segundos": [tiempo_pandas, tiempo_dask],
})

# Calculamos cuántas veces más lento es cada enfoque respecto al más rápido
comparacion["mas_lento_que_el_mejor"] = comparacion["tiempo_segundos"] / comparacion["tiempo_segundos"].min()
comparacion


,enfoque,tiempo_segundos,mas_lento_que_el_mejor
0,Pandas (secuencial),36.000367,1.333273
1,Dask (paralelo),27.001502,1.000000


En este experimento, Pandas y Dask tuvieron tiempos similares, e incluso Dask fue ligeramente más lento.  
Esto se debe principalmente al overhead (sobrecarga) de Dask para crear el task graph, administrar particiones y coordinar el procesamiento. Dask muestra sus mayores ventajas cuando los datasets son mucho más grandes, no caben en memoria o se distribuyen entre múltiples workers.

## 6. Filtros y agregaciones avanzadas con Dask

Ahora que ya sabemos como funciona el flujo basico, practiquemos con analisis un poco
mas ricos, tal como los harias en un caso real de negocio.


### 6.1 Cantidad de transacciones y monto total por canal y tipo de cliente

Vamos a agrupar por dos columnas a la vez, algo muy comun en reportes de negocio.


In [ ]:
resumen_canal_cliente = (  # Creamos un resumen agrupado por canal y tipo de cliente
    df_dask.groupby(["canal", "tipo_cliente"])["monto"]  # Agrupamos los datos por canal y tipo de cliente y seleccionamos monto
    .agg(["count", "sum", "mean"])  # Calculamos la cantidad de transacciones, el monto total y el monto promedio
    .compute()  # Ejecutamos el cálculo de Dask y obtenemos el resultado como un DataFrame de Pandas
    .sort_values("sum", ascending=False)  # Ordenamos los resultados de mayor a menor según el monto total

)

resumen_canal_cliente

,,count,sum,mean
canal,tipo_cliente,,,
web,personal,3458758,6.172757e+09,1784.674469
sucursal,personal,3457843,6.171374e+09,1784.746592
pos,personal,3457092,6.169848e+09,1784.692970
app_movil,personal,3452371,6.167077e+09,1786.330920
cajero_automatico,personal,3454257,6.166090e+09,1785.069849
app_movil,empresarial,1441230,2.570973e+09,1783.874488
web,empresarial,1439454,2.568392e+09,1784.282222
pos,empresarial,1439950,2.568271e+09,1783.583116
cajero_automatico,empresarial,1438730,2.568091e+09,1784.970543


### 6.2 Transacciones sospechosas: monto muy alto en canales digitales

Un analisis tipico de deteccion de fraude: transacciones de mas de 8000 dolares
hechas por `app_movil` o `web` (canales sin verificacion presencial).


In [ ]:
# Definimos la condición para identificar transacciones con montos altos realizadas por app móvil o web
condicion_sospecha = (df_dask["monto"] > 8000) & (df_dask["canal"].isin(["app_movil", "web"]))

transacciones_sospechosas = df_dask[condicion_sospecha].compute()  # Filtramos las transacciones y ejecutamos el cálculo de Dask para obtener un DataFrame de Pandas

print(f"Transacciones sospechosas encontradas: {len(transacciones_sospechosas):,}")
transacciones_sospechosas.sort_values("monto", ascending=False).head(10)

Transacciones sospechosas encontradas: 198,792


,id_transaccion,fecha,monto,region,canal,tipo_cliente
607519,25607519,2024-02-17,46204.63,La Libertad,web,personal
417544,23417544,2024-07-04,42554.55,La Libertad,app_movil,empresarial
139260,26139260,2024-11-23,36879.08,San Salvador,web,empresarial
179862,8179862,2024-02-01,36451.46,La Libertad,app_movil,empresarial
518693,15518693,2024-10-16,35290.53,La Libertad,app_movil,empresarial
453523,9453523,2024-11-18,35127.78,La Libertad,web,personal
619466,3019466,2024-02-26,34894.57,La Libertad,app_movil,pyme
404565,2004565,2024-08-21,34537.97,San Miguel,app_movil,personal
432044,9432044,2024-06-08,33856.05,La Libertad,web,personal
969350,12969350,2024-10-22,33449.50,La Libertad,web,empresarial


### 6.3 Distribucion de montos por region

Calculamos varias estadisticas descriptivas a la vez por region (cantidad, promedio,
desviacion estandar, minimo y maximo), usando `.agg()` con una lista de funciones,
igual que harias con Pandas, pero calculado en paralelo sobre todas las particiones.


In [ ]:
estadisticas_region = (  # Creamos un resumen estadístico agrupado por región

    df_dask.groupby("region")["monto"]  # Agrupamos los datos por región y seleccionamos la columna monto
    .agg(["count", "mean", "std", "min", "max"])  # Calculamos la cantidad, promedio, desviación estándar, mínimo y máximo de los montos
    .compute()  # Ejecutamos el cálculo de Dask y obtenemos el resultado
    .sort_values("mean", ascending=False)  # Ordenamos las regiones de mayor a menor según el monto promedio

)

estadisticas_region

,count,mean,std,min,max
region,,,,,
La Libertad,2877403,2804.970468,2799.377479,5.0,46204.63
San Salvador,2879677,2505.146441,2503.359388,5.0,36945.45
San Miguel,2880464,2202.763787,2196.525547,5.0,34537.97
Cuscatlan,2875902,2104.748702,2098.755864,5.0,30124.96
Santa Ana,2879907,1804.085256,1800.391525,5.0,30460.57
Sonsonate,2881004,1605.184748,1600.078566,5.0,24223.99
Usulutan,2880372,1403.917724,1399.833944,5.0,21616.08
La Paz,2881950,1305.261990,1301.109405,5.0,17881.06
Ahuachapan,2881530,1106.853435,1100.752785,5.0,17933.41


```df_dask``` es un Dask DataFrame porque fue creado con ```dd.read_csv()```.

Las operaciones como ```groupby()``` y ```agg()``` se mantienen en Dask y solo construyen el plan de ejecución.

```.compute()``` ejecuta realmente el procesamiento de Dask.

Por eso:

Antes de ```.compute()``` → Dask  
Después de ```.compute()``` → resultado como Pandas DataFrame

## 7. Errores comunes al trabajar con Dask

**1. Olvidar el `.compute()`**

Si imprimes una variable de Dask sin llamar a `.compute()`, no vas a ver los datos,
vas a ver la representacion del objeto (el "plan" de calculo, no el resultado).


In [ ]:
# Ejemplo del error: esto NO ejecuta el calculo, solo muestra el objeto Dask
promedio_sin_compute = df_dask.groupby("region")["monto"].mean()
print(promedio_sin_compute)


Dask Series Structure:
npartitions=1
    float64
        ...
Dask Name: getitem, 5 expressions
Expr=((ArrowStringConversion(frame=FromMapProjectable(04c4144))[['monto', 'region']]).mean(observed=False, chunk_kwargs={'numeric_only': False}, aggregate_kwargs={'numeric_only': False}, _slice='monto'))['monto']


**2. Demasiadas particiones para un archivo pequenio**

Si divides un dataset pequenio en cientos de particiones, el tiempo que Dask gasta
coordinando las tareas puede terminar siendo mayor que el tiempo real de
procesamiento. Como regla practica: usa Dask solo cuando el dataset realmente no cabe
comodo en RAM; para datasets pequenios, Pandas casi siempre gana.

**3. Mezclar tipos de datos entre particiones**

Dask infiere el tipo de cada columna leyendo solo una muestra de las primeras filas.
Si mas adelante en el archivo aparece un tipo de dato distinto (por ejemplo, una
columna que deberia ser numerica pero tiene un texto colado), el proceso puede fallar
o dar resultados inconsistentes. La solucion es declarar los tipos explicitamente con
el parametro `dtype`.


In [ ]:
# Ejemplo de como declarar los tipos explicitamente para evitar el error de inferencia
df_dask_tipado = dd.read_csv(
    os.path.join(DATA_DIR, "transacciones_2024_part*.csv"),
    dtype={
        "id_transaccion": "int64",
        "monto": "float64",
        "region": "object",
        "canal": "object",
        "tipo_cliente": "object",
    },
    parse_dates=["fecha"],
)
df_dask_tipado.dtypes


,0
id_transaccion,int64
fecha,datetime64[ns]
monto,float64
region,string[pyarrow]
canal,string[pyarrow]
tipo_cliente,string[pyarrow]


## 8. Exportar el resultado para la Sesion 2

En la proxima sesion vamos a llevar estos resultados a una base de datos NoSQL
(MongoDB). Para eso, vamos a guardar dos cosas:

1. El **reporte agregado** (promedio de monto por region, para transacciones de alto
   valor), que sera nuestro primer ejemplo de documentos "resumen" en Mongo.
2. Una **muestra de transacciones individuales de alto riesgo**, que sera nuestro
   ejemplo de documentos "detalle".

Guardamos ambos como CSV dentro de la carpeta `data/`, para que la Sesion 2 los pueda
leer directamente.


In [ ]:
# Ejecutamos el cálculo de Dask, ordenamos los promedios de mayor a menor y convertimos el índice en una col
reporte_regiones = promedio_por_region_dask.compute().sort_values(ascending=False).reset_index()
reporte_regiones.columns = ["region", "promedio_monto_alto_valor"] # Renombramos las columnas del reporte
reporte_regiones.to_csv(os.path.join(DATA_DIR, "reporte_promedio_por_region.csv"), index=False) # Guardamos el reporte como archivo CSV

# Filtramos transacciones con monto mayor a 8000, ejecutamos Dask y tomamos una muestra aleatoria
muestra_alto_riesgo = df_dask[df_dask["monto"] > 8000].compute().sample(
    n=min(500, len(transacciones_sospechosas) + 1000), random_state=42 # Seleccionamos como máximo 500 registros
)
muestra_alto_riesgo.to_csv(os.path.join(DATA_DIR, "muestra_transacciones_alto_riesgo.csv"), index=False) # Guardamos la muestra de transacciones como archivo CSV

print("Archivos guardados para la Sesion 2:")
print(" -", os.path.join(DATA_DIR, "reporte_promedio_por_region.csv"))
print(" -", os.path.join(DATA_DIR, "muestra_transacciones_alto_riesgo.csv"))


Archivos guardados para la Sesion 2:
 - data/reporte_promedio_por_region.csv
 - data/muestra_transacciones_alto_riesgo.csv


## Cierre de la sesion

Hoy aprendimos:

- Por que las herramientas de procesamiento secuencial (Pandas puro) se vuelven un
  cuello de botella cuando el dataset crece.
- Como Dask usa *lazy evaluation* y particiones para escalar el mismo tipo de
  analisis que ya sabias hacer con Pandas.
- Como comparar el rendimiento de ambos enfoques de forma objetiva, midiendo tiempo.
- Los errores mas comunes al empezar con Dask y como evitarlos.

### Reto opcional para practicar

Usando `df_dask`, calcula cual es el dia de la semana con mayor monto total de
transacciones (pista: puedes extraer el dia de la semana de la columna `fecha` con
`.dt.day_name()` antes de agrupar). Compara cuanto tarda tu solucion con Dask contra
la misma logica implementada en Pandas puro.

En la Sesion 2 vamos a tomar los archivos que guardamos en `data/` y los vamos a
cargar a una base de datos NoSQL (MongoDB) para practicar consultas mas alla del
CSV.
